In [ ]:
import numpy as np
import os 
BASE_DIR = "../data"
CONN_DIR = os.path.join(BASE_DIR, "fMRI_data/subject_windows_confounds_old_60sec")

list_of_files = os.listdir(CONN_DIR)
print(list_of_files)

all_data = []
all_labels = []
for file in list_of_files:
    sub_data = []
    if file.endswith(".npz"):
        data = np.load(os.path.join(CONN_DIR, file))
        
        AP_temp = data["AP"]
        PA_temp = data["PA"]
        label = data["label"]
        
        
        sub_data = np.concatenate((AP_temp, PA_temp), axis=0)
        
        all_data.append(sub_data)
        all_labels.append(label)

all_data = np.array(all_data)
all_labels = np.array(all_labels)

print(all_data.shape)
print(all_labels.shape)
        


In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import numpy as np

KMEANS_RANDOM_STATE = 1
KMEANS_N_INIT = "auto"
KMEANS_MAX_ITER = 100000
PCA_RANDOM_STATE = 1

X_scaled = all_data.reshape(-1, all_data.shape[-1])

# --- Elbow Method Calculation ---
k_values = range(2, 15)
inertia_list = []

print("Calculating inertia for each K value...")
for k in k_values:
    kmeans_model = KMeans(
        n_clusters=k,
        random_state=KMEANS_RANDOM_STATE,
        n_init=KMEANS_N_INIT,
        max_iter=KMEANS_MAX_ITER
    )
    kmeans_model.fit(X_scaled)
    inertia_list.append(kmeans_model.inertia_)
    print(f"  K={k}, Inertia={kmeans_model.inertia_:.2f}")
print("Calculation complete.")


#

In [ ]:
from kneed import KneeLocator# --- Elbow Point Detection ---
knee_locator = KneeLocator(
    k_values, inertia_list, curve='convex', direction='decreasing'
)
optimal_k = knee_locator.knee
print(f"Optimal number of clusters (K) determined: {optimal_k}")

In [ ]:
# import warnings
# import pandas as pd
# import numpy as np
# from pathlib import Path
# from scipy import stats
# from statsmodels.stats.multitest import multipletests
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
# from statsmodels.stats.multicomp import pairwise_tukeyhsd

# # Statistical tests (both parametric and non-parametric)
# from scipy.stats import kruskal, mannwhitneyu, shapiro, levene

# # ----------------- USER/PLOTTING CONFIG -----------------
# K_RANGE = range(5, 11)
# WIN_STEP = 5
# TR = 0.8
# KMEANS_RANDOM_STATE = 9
# KMEANS_N_INIT = "auto"
# KMEANS_MAX_ITER = 10000
# PCA_RANDOM_STATE = 0
# P_VALUE_THRESHOLD = 0.05 # Uncorrected p-value threshold for initial state selection
# POSTHOC_ALPHA = 0.05 # Alpha for Tukey's HSD

# # Statistical test parameters
# NORMALITY_ALPHA = 0.05  # Significance level for normality tests
# VARIANCE_ALPHA = 0.05   # Significance level for variance homogeneity test
# MIN_SAMPLE_SIZE = 3     # Minimum samples per group for normality testing

# # ---- Sanity checks ----
# if "all_data" not in globals() or "all_labels" not in globals():
#     raise RuntimeError("Please define `all_data` and `all_labels` before running.")
# if np.isnan(all_data).any():
#     raise ValueError("all_data contains NaNs. Please clean or impute before running.")

# n_subjects, n_timepoints, n_features = all_data.shape
# print(f"Data: {n_subjects} subjects × {n_timepoints} timepoints × {n_features} features")
# print("Using statistical tests with normality and variance checks (ANOVA/F-test vs Mann-Whitney U / Kruskal-Wallis)")

# # ---------------- PREP DATA ----------------
# reshaped_data = all_data.reshape(-1, n_features)
# X_scaled = reshaped_data
# print(f"Reshaped data for clustering / PCA: {reshaped_data.shape}")

# # Store statistical test results
# all_statistical_results = []

# def test_normality_and_variance(groups, state_id, k_clusters):
#     """
#     Test normality and homogeneity of variance for groups.
#     Returns: (use_parametric, test_details)
#     """
#     test_details = {
#         'state': state_id,
#         'k': k_clusters,
#         'n_groups': len(groups),
#         'group_sizes': [len(g) for g in groups],
#         'normality_test': 'Not performed',
#         'variance_test': 'Not performed',
#         'use_parametric': False,
#         'reason': ''
#     }
    
#     # Check if we have enough data for normality testing
#     if any(len(g) < MIN_SAMPLE_SIZE for g in groups):
#         test_details['reason'] = f'Sample size too small (min {MIN_SAMPLE_SIZE} required)'
#         test_details['use_parametric'] = False
#         return False, test_details
    
#     # Test normality for each group using Shapiro-Wilk
#     normality_pvals = []
#     for i, group in enumerate(groups):
#         if len(group) >= 3:  # Shapiro-Wilk requires at least 3 samples
#             try:
#                 _, p_val = shapiro(group)
#                 normality_pvals.append(p_val)
#             except:
#                 normality_pvals.append(0.0)  # Assume non-normal if test fails
#         else:
#             normality_pvals.append(0.0)  # Too small for normality test
    
#     # Check if all groups are normally distributed
#     all_normal = all(p > NORMALITY_ALPHA for p in normality_pvals)
#     test_details['normality_test'] = f"Shapiro-Wilk p-values: {[f'{p:.4f}' for p in normality_pvals]}"
    
#     if not all_normal:
#         test_details['reason'] = 'One or more groups not normally distributed'
#         test_details['use_parametric'] = False
#         return False, test_details
    
#     # Test homogeneity of variance using Levene's test
#     try:
#         _, variance_p = levene(*groups)
#         test_details['variance_test'] = f"Levene's test p-value: {variance_p:.4f}"
        
#         if variance_p <= VARIANCE_ALPHA:
#             test_details['reason'] = 'Unequal variances detected'
#             test_details['use_parametric'] = False
#             return False, test_details
        
#     except:
#         test_details['variance_test'] = 'Levene test failed'
#         test_details['reason'] = 'Could not test variance homogeneity'
#         test_details['use_parametric'] = False
#         return False, test_details
    
#     # All assumptions met
#     test_details['reason'] = 'All parametric assumptions satisfied'
#     test_details['use_parametric'] = True
#     return True, test_details

# def choose_statistical_test(groups, state_id, k_clusters):
#     """
#     Choose appropriate statistical test based on normality and variance assumptions.
#     Returns: (test_type, p_value, f_statistic, test_details)
#     """
#     # Basic checks
#     if len(groups) < 2 or any(g.size == 0 for g in groups):
#         test_details = {
#             'state': state_id, 'k': k_clusters, 'n_groups': len(groups),
#             'group_sizes': [len(g) for g in groups], 'test_used': 'None',
#             'reason': 'Insufficient groups or empty groups', 'use_parametric': False
#         }
#         return 'None', 1.0, 0.0, test_details
    
#     if np.std(np.concatenate(groups)) <= 1e-10:
#         test_details = {
#             'state': state_id, 'k': k_clusters, 'n_groups': len(groups),
#             'group_sizes': [len(g) for g in groups], 'test_used': 'None',
#             'reason': 'No variance in data', 'use_parametric': False
#         }
#         return 'None', 1.0, 0.0, test_details
    
#     # Check if any group has zero variance (all identical values)
#     group_stds = [np.std(g) for g in groups]
#     if any(std <= 1e-10 for std in group_stds):
#         test_details = {
#             'state': state_id, 'k': k_clusters, 'n_groups': len(groups),
#             'group_sizes': [len(g) for g in groups], 'test_used': 'None',
#             'reason': 'One or more groups has no variance', 'use_parametric': False
#         }
#         return 'None', 1.0, 0.0, test_details
    
#     # Test assumptions and choose appropriate test
#     use_parametric, test_details = test_normality_and_variance(groups, state_id, k_clusters)
    
#     try:
#         if use_parametric:
#             # Use parametric tests (ANOVA for >2 groups, t-test for 2 groups)
#             if len(groups) == 2:
#                 # For 2 groups, we could use independent t-test, but f_oneway works too
#                 f_stat, p_value = stats.f_oneway(*groups)
#                 test_details['test_used'] = 'F-test/ANOVA'
#                 print(f"  State {state_id}: Using F-test (parametric assumptions met)")
#             else:
#                 # Use ANOVA (F-test) for >2 groups
#                 f_stat, p_value = stats.f_oneway(*groups)
#                 test_details['test_used'] = 'ANOVA'
#                 print(f"  State {state_id}: Using ANOVA (parametric assumptions met)")
            
#             return test_details['test_used'], p_value, f_stat, test_details
#         else:
#             # Use non-parametric tests
#             if len(groups) == 2:
#                 # Use Mann-Whitney U test for 2 groups
#                 statistic, p_value = mannwhitneyu(groups[0], groups[1], alternative='two-sided')
#                 test_details['test_used'] = 'Mann-Whitney'
#                 print(f"  State {state_id}: Using Mann-Whitney U ({test_details['reason']})")
#                 return 'Mann-Whitney', p_value, statistic, test_details
#             else:
#                 # Use Kruskal-Wallis for >2 groups
#                 statistic, p_value = kruskal(*groups)
#                 test_details['test_used'] = 'Kruskal-Wallis'
#                 print(f"  State {state_id}: Using Kruskal-Wallis ({test_details['reason']})")
#                 return 'Kruskal-Wallis', p_value, statistic, test_details
                
#     except Exception as e:
#         test_details['reason'] = f'Statistical test failed: {str(e)}'
#         test_details['test_used'] = 'None'
#         print(f"Warning: Statistical test failed for state {state_id}: {e}")
#         return 'None', 1.0, 0.0, test_details

# # ---- Loop through all K values ----
# for K_CLUSTERS in K_RANGE:
#     print(f"\n{'='*80}")
#     print(f"RUNNING ANALYSIS FOR K = {K_CLUSTERS}")
#     print(f"{'='*80}")

#     # ---------------- KMEANS ----------------
#     kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=KMEANS_RANDOM_STATE, 
#                    n_init=KMEANS_N_INIT, max_iter=KMEANS_MAX_ITER)
#     cluster_labels_flat = kmeans.fit_predict(X_scaled)
#     cluster_labels = cluster_labels_flat.reshape(n_subjects, n_timepoints)

#     # ---- Dwell-time extraction ----
#     def extract_dwell_time_features_for_subjects(cluster_labels_reshaped, k_clusters, win_step, tr):
#         dwell = np.zeros((cluster_labels_reshaped.shape[0], k_clusters), dtype=float)
#         for s in range(cluster_labels_reshaped.shape[0]):
#             counts = np.bincount(cluster_labels_reshaped[s], minlength=k_clusters)
#             dwell[s, :] = counts * win_step * tr
#         return dwell

#     all_dwell_times = extract_dwell_time_features_for_subjects(cluster_labels, K_CLUSTERS, WIN_STEP, TR)

#     # ----------------- STATISTICAL TESTING and state selection -----------------
#     unique_labels = np.unique(all_labels)
#     all_states_stats = []
    
#     for state in range(K_CLUSTERS):
#         groups = [all_dwell_times[all_labels == label, state] for label in unique_labels]
#         groups = [g for g in groups if g.size > 0]  # Remove empty groups
        
#         # Choose appropriate statistical test based on assumptions
#         test_type, p_value, f_stat, test_details = choose_statistical_test(
#             groups, state, K_CLUSTERS)
        
#         # Store statistical results
#         all_statistical_results.append(test_details)
        
#         if test_type != 'None':
#             # Calculate means for coloring purposes
#             mean_g0 = np.mean(groups[0]) if groups[0].size > 0 else 0
#             mean_g1 = np.mean(groups[1]) if len(groups) > 1 and groups[1].size > 0 else 0
            
#             all_states_stats.append({
#                 'state': state, 
#                 'p_value': p_value, 
#                 'f_statistic': f_stat,
#                 'test_type': test_type,
#                 'mean_g0': mean_g0, 
#                 'mean_g1': mean_g1
#             })
    
#     if not all_states_stats:
#         print("No valid statistical tests performed for this K, skipping.")
#         continue
    
#     # Sort all states by p-value
#     all_states_stats.sort(key=lambda x: x['p_value'])
    
#     # Select states to plot
#     significant_states = [s for s in all_states_stats if s['p_value'] < P_VALUE_THRESHOLD]
#     top_states_to_plot = significant_states[:2]

#     # For specific K values, ensure we have at least 2 states to plot
#     if len(top_states_to_plot) < 2 and len(all_states_stats) >= 2:
#         # Add the next best states (even if not significant)
#         top_states_to_plot = all_states_stats[:2]

#     print(f"K={K_CLUSTERS}: Found {len(significant_states)} significant states, plotting top {len(top_states_to_plot)}.")
    
#     if not top_states_to_plot:
#         print("No states to plot for this K, skipping figure generation.")
#         continue

#     # ----------------- SETUP COMBINED PLOT -----------------
#     n_to_plot = len(top_states_to_plot)
#     fig = plt.figure(figsize=(8 + 6 * n_to_plot, 7))
#     gs = gridspec.GridSpec(1, 1 + n_to_plot, figure=fig, width_ratios=[8, *([6]*n_to_plot)])

#     # ----------------- 1. PCA PLOT (LEFT) -----------------
#     ax_pca = fig.add_subplot(gs[0, 0])
#     pca = PCA(n_components=2, random_state=PCA_RANDOM_STATE)
#     X2 = pca.fit_transform(X_scaled)
#     centers2 = pca.transform(kmeans.cluster_centers_)
    
#     # Color scheme based on mean differences
#     cluster_colors = {res['state']: ('royalblue' if res['mean_g0'] > res['mean_g1'] else 'crimson') 
#                      for res in top_states_to_plot}
#     point_colors = [cluster_colors.get(l, '#D3D3D3') for l in cluster_labels_flat]
    
#     ax_pca.scatter(X2[:, 0], X2[:, 1], c=point_colors, s=15, alpha=0.4)
#     ax_pca.scatter(centers2[:, 0], centers2[:, 1], marker='X', s=150, c='k', 
#                   edgecolors='white', linewidth=0.5)
    
#     for idx, (x, y) in enumerate(centers2):
#         color = cluster_colors.get(idx, 'gray')
#         bbox_style = dict(facecolor=color, alpha=0.8, boxstyle='round,pad=0.2')
#         ax_pca.text(x, y, f"{idx}", color='w', fontsize=18, ha='center', va='center', bbox=bbox_style)

#     ax_pca.set_xlabel('Principal Component 1', fontsize=18)
#     ax_pca.set_ylabel('Principal Component 2', fontsize=18)
#     ax_pca.set_title(f'PCA of K-Means States (K={K_CLUSTERS})', fontsize=18)
#     ax_pca.grid(True, linestyle='--', alpha=0.5)
#     ax_pca.set_facecolor('white')
#     for spine in ['top', 'right']:
#         ax_pca.spines[spine].set_visible(False)
#     for spine in ['left', 'bottom']:
#         ax_pca.spines[spine].set_color('gray')

#     # ----------------- 2. BOX PLOTS (RIGHT) WITH POST-HOC TESTING -----------------
#     label_mapping = {0: 'N', 1: 'A+P-', 2: 'A+P+'}
#     group_colors = {'N': 'royalblue', 'A+P-': 'crimson', 'A+P+': 'seagreen'}

#     for i, result in enumerate(top_states_to_plot):
#         ax_box = fig.add_subplot(gs[0, 1 + i])
#         state = result['state']
        
#         plot_data = [all_dwell_times[all_labels == label, state] for label in unique_labels]
#         plot_labels = [label_mapping.get(label, f'Group {label}') for label in unique_labels]
        
#         # Create boxplot
#         bp = ax_box.boxplot(plot_data, patch_artist=True, labels=plot_labels,
#                           boxprops=dict(linewidth=1.5), whiskerprops=dict(linewidth=1.5),
#                           capprops=dict(linewidth=1.5), medianprops=dict(linewidth=2, color='black'))
        
#         for patch, label in zip(bp['boxes'], plot_labels):
#             patch.set_facecolor(group_colors.get(label, 'gray'))
#             patch.set_alpha(0.6)

#         # Add individual data points with jitter
#         for j, data in enumerate(plot_data):
#             x_jitter = np.random.normal(j + 1, 0.04, size=len(data))
#             ax_box.scatter(x_jitter, data, alpha=0.6, s=30, color='k', zorder=3)

#         # Create title with test information
#         test_type = result['test_type']
#         p_val = result['p_value']
#         sig_level = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else ''))
#         title_text = f'State {state}\n({test_type} p={p_val:.3f} {sig_level})'
        
#         ax_box.set_title(title_text, fontsize=18)
#         ax_box.set_ylabel('Dwell Time (seconds)', fontsize=18)
#         ax_box.tick_params(axis='x', labelsize=16)
#         ax_box.grid(True, alpha=0.3, axis='y')
#         ax_box.set_facecolor('white')
#         for spine in ['top', 'right']:
#             ax_box.spines[spine].set_visible(False)
#         for spine in ['left', 'bottom']:
#             ax_box.spines[spine].set_color('gray')

#         # --- Post-hoc Testing (only if main test is significant) ---
#         if p_val < P_VALUE_THRESHOLD and len(np.unique(plot_labels)) > 2:
#             # Prepare data for post-hoc test
#             tukey_data = np.concatenate(plot_data)
#             tukey_labels = np.concatenate([[label] * len(d) for label, d in zip(plot_labels, plot_data)])
            
#             try:
#                 # Use Tukey's HSD for post-hoc comparisons
#                 tukey_result = pairwise_tukeyhsd(endog=tukey_data, groups=tukey_labels, alpha=POSTHOC_ALPHA)
#                 tukey_df = pd.DataFrame(data=tukey_result._results_table.data[1:], 
#                                       columns=tukey_result._results_table.data[0])
                
#                 # Plot significance bars
#                 y_max = max([d.max() for d in plot_data if d.size > 0])
#                 y_pos = y_max * 1.05
#                 y_step = y_max * 0.08
                
#                 for _, row in tukey_df.iterrows():
#                     if row['reject']:  # If the pair is significant
#                         group1, group2 = row['group1'], row['group2']
#                         p_adj = row['p-adj']
                        
#                         sig_level = '***' if p_adj < 0.001 else ('**' if p_adj < 0.01 else '*')

#                         try:
#                             x1 = plot_labels.index(group1) + 1
#                             x2 = plot_labels.index(group2) + 1
                            
#                             # Plot significance bar
#                             ax_box.plot([x1, x2], [y_pos, y_pos], 'k-', linewidth=1.5)
#                             ax_box.text((x1 + x2) / 2, y_pos, sig_level, ha='center', va='bottom', fontsize=16)
#                             y_pos += y_step
#                         except ValueError:
#                             continue
                            
#             except Exception as e:
#                 print(f"    Warning: Post-hoc test failed for state {state}: {e}")

#     # --- Finalize and Save ---
#     fig.tight_layout(pad=2.0)
    
#     OUTFIG_COMBINED = Path(f'combined_analysis_k={K_CLUSTERS}_adaptive_tests.png')
#     plt.savefig(OUTFIG_COMBINED, dpi=300, bbox_inches='tight', facecolor='white')
#     plt.show()

# # ----------------- SUMMARY OF STATISTICAL TESTS USED -----------------
# print(f"\n\n{'='*80}")
# print("SUMMARY OF STATISTICAL TESTS USED (ADAPTIVE APPROACH)")
# print(f"{'='*80}")

# if all_statistical_results:
#     # Create summary DataFrame
#     test_summary_df = pd.DataFrame(all_statistical_results)
    
#     # Count test types used
#     test_counts = test_summary_df['test_used'].value_counts()
    
#     print(f"\nOverall Test Usage:")
#     for test_type, count in test_counts.items():
#         print(f"  {test_type}: {count} states")
    
#     # Count parametric vs non-parametric
#     parametric_count = test_summary_df['use_parametric'].sum()
#     nonparametric_count = len(test_summary_df) - parametric_count
    
#     print(f"\nParametric vs Non-parametric breakdown:")
#     print(f"  Parametric tests (ANOVA/F-test): {parametric_count} states")
#     print(f"  Non-parametric tests (Mann-Whitney/Kruskal-Wallis): {nonparametric_count} states")
    
#     print(f"\nReasons for non-parametric tests:")
#     nonparametric_df = test_summary_df[~test_summary_df['use_parametric']]
#     if not nonparametric_df.empty:
#         reason_counts = nonparametric_df['reason'].value_counts()
#         for reason, count in reason_counts.items():
#             print(f"  {reason}: {count} states")
    
#     # Show detailed breakdown by K
#     print(f"\nDetailed breakdown by K value:")
#     for k in K_RANGE:
#         k_data = test_summary_df[test_summary_df['k'] == k]
#         if not k_data.empty:
#             anova_count = ((k_data['test_used'] == 'ANOVA') | (k_data['test_used'] == 'F-test/ANOVA')).sum()
#             mann_whitney_count = (k_data['test_used'] == 'Mann-Whitney').sum()
#             kruskal_wallis_count = (k_data['test_used'] == 'Kruskal-Wallis').sum()
#             none_count = (k_data['test_used'] == 'None').sum()
#             total_k = len(k_data)
            
#             print(f"  K={k}: {anova_count} ANOVA/F-test, {mann_whitney_count} Mann-Whitney, {kruskal_wallis_count} Kruskal-Wallis, {none_count} none (out of {total_k} total)")

#     # Calculate percentage of successful tests
#     successful_tests = test_summary_df[test_summary_df['test_used'] != 'None']
#     if len(successful_tests) > 0:
#         success_rate = len(successful_tests) / len(test_summary_df) * 100
#         print(f"\nOverall success rate: {success_rate:.1f}% of state comparisons used valid statistical tests")
    
#     # Save detailed results
#     test_summary_df.to_csv('combined_analysis_statistical_tests_adaptive.csv', index=False)
#     print(f"\nDetailed test results saved to 'combined_analysis_statistical_tests_adaptive.csv'")

# print("\n\nCombined analysis complete with adaptive statistical testing!")
# print("✓ Used parametric tests when assumptions were met, non-parametric otherwise")
# print("✓ Included normality testing (Shapiro-Wilk) and variance homogeneity testing (Levene's test)")

In [ ]:
# import warnings
# import pandas as pd
# import numpy as np
# from pathlib import Path
# from scipy import stats
# from statsmodels.stats.multitest import multipletests
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
# from statsmodels.stats.multicomp import pairwise_tukeyhsd
# import os

# # Statistical tests (both parametric and non-parametric)
# from scipy.stats import kruskal, mannwhitneyu, shapiro, levene

# # ----------------- USER/PLOTTING CONFIG -----------------
# K_RANGE = range(5, 11)
# WIN_STEP = 5
# TR = 0.8
# SEEDS = [100]  # Five different seeds
# KMEANS_N_INIT = "auto"
# KMEANS_MAX_ITER = 10000
# PCA_RANDOM_STATE = 0
# FDR_ALPHA = 0.05  # For Benjamini-Hochberg FDR correction
# POSTHOC_ALPHA = 0.05  # For post-hoc Bonferroni correction

# # Statistical test parameters
# NORMALITY_ALPHA = 0.05
# VARIANCE_ALPHA = 0.05
# MIN_SAMPLE_SIZE = 3

# # Create output directory
# OUTPUT_DIR = Path("fMRI_windows_analysis")
# OUTPUT_DIR.mkdir(exist_ok=True)

# # ---- Sanity checks ----
# if "all_data" not in globals() or "all_labels" not in globals():
#     raise RuntimeError("Please define `all_data` and `all_labels` before running.")
# if np.isnan(all_data).any():
#     raise ValueError("all_data contains NaNs. Please clean or impute before running.")

# n_subjects, n_timepoints, n_features = all_data.shape
# print(f"Data: {n_subjects} subjects × {n_timepoints} timepoints × {n_features} features")
# print("Using Benjamini-Hochberg FDR correction across ALL p-values and K values")

# # ---------------- PREP DATA ----------------
# reshaped_data = all_data.reshape(-1, n_features)
# X_scaled = reshaped_data
# print(f"Reshaped data for clustering / PCA: {reshaped_data.shape}")

# def test_normality_and_variance(groups, state_id, k_clusters):
#     """Test normality and homogeneity of variance for groups."""
#     test_details = {
#         'state': state_id,
#         'k': k_clusters,
#         'n_groups': len(groups),
#         'group_sizes': [len(g) for g in groups],
#         'normality_test': 'Not performed',
#         'variance_test': 'Not performed',
#         'use_parametric': False,
#         'reason': ''
#     }
    
#     if any(len(g) < MIN_SAMPLE_SIZE for g in groups):
#         test_details['reason'] = f'Sample size too small (min {MIN_SAMPLE_SIZE} required)'
#         return False, test_details
    
#     # Test normality using Shapiro-Wilk
#     normality_pvals = []
#     for group in groups:
#         if len(group) >= 3:
#             try:
#                 _, p_val = shapiro(group)
#                 normality_pvals.append(p_val)
#             except:
#                 normality_pvals.append(0.0)
#         else:
#             normality_pvals.append(0.0)
    
#     all_normal = all(p > NORMALITY_ALPHA for p in normality_pvals)
#     test_details['normality_test'] = f"Shapiro-Wilk p-values: {[f'{p:.4f}' for p in normality_pvals]}"
    
#     if not all_normal:
#         test_details['reason'] = 'One or more groups not normally distributed'
#         return False, test_details
    
#     # Test variance homogeneity using Levene's test
#     try:
#         _, variance_p = levene(*groups)
#         test_details['variance_test'] = f"Levene's test p-value: {variance_p:.4f}"
        
#         if variance_p <= VARIANCE_ALPHA:
#             test_details['reason'] = 'Unequal variances detected'
#             return False, test_details
#     except:
#         test_details['variance_test'] = 'Levene test failed'
#         test_details['reason'] = 'Could not test variance homogeneity'
#         return False, test_details
    
#     test_details['reason'] = 'All parametric assumptions satisfied'
#     test_details['use_parametric'] = True
#     return True, test_details

# def choose_statistical_test(groups, state_id, k_clusters):
#     """Choose appropriate statistical test based on assumptions."""
#     # Basic checks
#     if len(groups) < 2 or any(g.size == 0 for g in groups):
#         test_details = {
#             'state': state_id, 'k': k_clusters, 'test_used': 'None',
#             'reason': 'Insufficient groups or empty groups'
#         }
#         return 'None', 1.0, 0.0, test_details
    
#     if np.std(np.concatenate(groups)) <= 1e-10:
#         test_details = {
#             'state': state_id, 'k': k_clusters, 'test_used': 'None',
#             'reason': 'No variance in data'
#         }
#         return 'None', 1.0, 0.0, test_details
    
#     group_stds = [np.std(g) for g in groups]
#     if any(std <= 1e-10 for std in group_stds):
#         test_details = {
#             'state': state_id, 'k': k_clusters, 'test_used': 'None',
#             'reason': 'One or more groups has no variance'
#         }
#         return 'None', 1.0, 0.0, test_details
    
#     # Test assumptions
#     use_parametric, test_details = test_normality_and_variance(groups, state_id, k_clusters)
    
#     try:
#         if use_parametric:
#             f_stat, p_value = stats.f_oneway(*groups)
#             test_details['test_used'] = 'ANOVA' if len(groups) > 2 else 'F-test'
#             return test_details['test_used'], p_value, f_stat, test_details
#         else:
#             if len(groups) == 2:
#                 statistic, p_value = mannwhitneyu(groups[0], groups[1], alternative='two-sided')
#                 test_details['test_used'] = 'Mann-Whitney'
#                 return 'Mann-Whitney', p_value, statistic, test_details
#             else:
#                 statistic, p_value = kruskal(*groups)
#                 test_details['test_used'] = 'Kruskal-Wallis'
#                 return 'Kruskal-Wallis', p_value, statistic, test_details
#     except Exception as e:
#         test_details['reason'] = f'Statistical test failed: {str(e)}'
#         test_details['test_used'] = 'None'
#         return 'None', 1.0, 0.0, test_details

# def compute_std_distance_to_centroid(cluster_labels_flat, X_data, kmeans_model, n_subjects, n_timepoints):
#     """Compute std_distance_to_centroid for each subject and state."""
#     k_clusters = kmeans_model.n_clusters
#     std_distances = np.full((n_subjects, k_clusters), np.nan)
    
#     for s in range(n_subjects):
#         for k in range(k_clusters):
#             # Get windows for this subject in this cluster
#             mask = (cluster_labels_flat == k) & (np.arange(len(cluster_labels_flat)) // n_timepoints == s)
#             cluster_points = X_data[mask]
            
#             if cluster_points.shape[0] > 0:
#                 # Compute distances to centroid
#                 distances = np.linalg.norm(cluster_points - kmeans_model.cluster_centers_[k], axis=1)
#                 std_distances[s, k] = np.std(distances)
    
#     return std_distances

# # Store ALL results for FDR correction (ACROSS ALL K and SEEDS)
# all_test_results = []

# # ---- Loop through all seeds ----
# for SEED in SEEDS:
#     print(f"\n{'='*100}")
#     print(f"PROCESSING SEED {SEED}")
#     print(f"{'='*100}")
    
#     # ---- Loop through all K values for this seed ----
#     for K_CLUSTERS in K_RANGE:
#         print(f"\n{'-'*60}")
#         print(f"SEED {SEED}: RUNNING ANALYSIS FOR K = {K_CLUSTERS}")
#         print(f"{'-'*60}")

#         # ---------------- KMEANS ----------------
#         kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=SEED, 
#                        n_init=KMEANS_N_INIT, max_iter=KMEANS_MAX_ITER)
#         cluster_labels_flat = kmeans.fit_predict(X_scaled)
#         cluster_labels = cluster_labels_flat.reshape(n_subjects, n_timepoints)

#         # Compute std_distance_to_centroid features
#         all_std_distances = compute_std_distance_to_centroid(
#             cluster_labels_flat, X_scaled, kmeans, n_subjects, n_timepoints
#         )
        
#         # ----------------- STATISTICAL TESTING for each state -----------------
#         unique_labels = np.unique(all_labels)
        
#         for state in range(K_CLUSTERS):
#             vals = all_std_distances[:, state]
            
#             # Handle NaN values
#             valid_mask = ~np.isnan(vals)
#             if not np.any(valid_mask):
#                 continue
            
#             valid_vals = vals[valid_mask]
#             valid_labels = all_labels[valid_mask]
#             valid_subject_ids = np.where(valid_mask)[0]
            
#             if len(np.unique(valid_labels)) < 2:
#                 continue
            
#             # Create groups for statistical testing
#             groups = [valid_vals[valid_labels == label] for label in unique_labels]
#             groups = [g for g in groups if g.size > 0]
            
#             # Statistical test
#             test_type, p_value, f_stat, test_details = choose_statistical_test(groups, state, K_CLUSTERS)
            
#             if test_type != 'None':
#                 # Calculate group means
#                 group_means = {label: np.mean(valid_vals[valid_labels == label]) 
#                               for label in unique_labels if np.any(valid_labels == label)}
                
#                 # Determine dominant group
#                 dominant_label = max(group_means, key=group_means.get)
#                 label_to_group = {0: 'N', 1: 'A+P-', 2: 'A+P+'}
#                 dominant_group = label_to_group.get(dominant_label, f'Group{dominant_label}')
                
#                 # Store result (before correction) - STORE RAW DATA FOR POST-HOC
#                 all_test_results.append({
#                     'seed': SEED,
#                     'K': K_CLUSTERS,
#                     'feature_type': 'std_distance_to_centroid',
#                     'state': state,
#                     'p_value': p_value,
#                     'test': test_type,
#                     'dominant_group': dominant_group,
#                     'group_means': group_means,
#                     'feature_values': vals,  # Include ALL values (with NaN)
#                     'valid_feature_values': valid_vals,  # Only valid values
#                     'valid_labels': valid_labels,  # Corresponding labels
#                     'cluster_labels': cluster_labels_flat,
#                     'kmeans_centers': kmeans.cluster_centers_,
#                     'subject_ids': valid_subject_ids
#                 })
                
#                 print(f"  State {state}: {test_type} p={p_value:.6f} ({dominant_group} higher)")

# # =============== BENJAMINI-HOCHBERG FDR CORRECTION ACROSS ALL P-VALUES ===============
# print(f"\n{'='*100}")
# print("APPLYING BENJAMINI-HOCHBERG FDR CORRECTION ACROSS ALL K VALUES AND SEEDS")
# print(f"{'='*100}")

# if all_test_results:
#     # Extract all p-values
#     p_values = [feat['p_value'] for feat in all_test_results]
    
#     # Apply Benjamini-Hochberg FDR correction
#     reject, pvals_corrected, _, _ = multipletests(p_values, alpha=FDR_ALPHA, method='fdr_bh')
    
#     # Add corrected p-values to results
#     for i, feat in enumerate(all_test_results):
#         feat['p_value_fdr_corrected'] = pvals_corrected[i]
#         feat['significant_after_fdr'] = reject[i]
    
#     print(f"\nTotal tests performed: {len(all_test_results)}")
#     print(f"Significant after FDR correction (α = {FDR_ALPHA}): {sum(reject)}")
    
#     print("\n✓ Significant states after Benjamini-Hochberg FDR correction:")
#     for feat in all_test_results:
#         if feat['significant_after_fdr']:
#             print(f"   Seed={feat['seed']}, K={feat['K']}, State {feat['state']}: "
#                   f"p_raw={feat['p_value']:.6f}, p_FDR={feat['p_value_fdr_corrected']:.6f}")

#     # Filter to significant results only
#     significant_results = [f for f in all_test_results if f['significant_after_fdr']]
    
#     # =============== POST-HOC COMPARISONS FOR SIGNIFICANT STATES ===============
#     print(f"\n{'='*100}")
#     print("CONDUCTING POST-HOC PAIRWISE COMPARISONS (BONFERRONI-CORRECTED)")
#     print(f"{'='*100}")
    
#     posthoc_results = []
    
#     for feat in significant_results:
#         SEED = feat['seed']
#         K_CLUSTERS = feat['K']
#         state = feat['state']
#         valid_vals = feat['valid_feature_values']
#         valid_labels = feat['valid_labels']
        
#         print(f"\nPost-hoc for Seed={SEED}, K={K_CLUSTERS}, State={state}:")
        
#         # Create dataframe for Tukey HSD
#         label_map = {0: 'N', 1: 'A+P-', 2: 'A+P+'}
#         group_labels = [label_map[l] for l in valid_labels]
        
#         try:
#             # Perform Tukey's HSD (automatically Bonferroni-corrected for multiple comparisons)
#             tukey_result = pairwise_tukeyhsd(endog=valid_vals, groups=group_labels, alpha=POSTHOC_ALPHA)
            
#             # Extract results
#             tukey_df = pd.DataFrame(data=tukey_result._results_table.data[1:], 
#                                   columns=tukey_result._results_table.data[0])
            
#             # Store results
#             for _, row in tukey_df.iterrows():
#                 posthoc_results.append({
#                     'seed': SEED,
#                     'K': K_CLUSTERS,
#                     'state': state,
#                     'group1': row['group1'],
#                     'group2': row['group2'],
#                     'meandiff': row['meandiff'],
#                     'lower_ci': row['lower'],
#                     'upper_ci': row['upper'],
#                     'reject': row['reject'],
#                     'p_adj': row['p-adj']
#                 })
                
#                 sig_marker = '*' if row['reject'] else ''
#                 print(f"  {row['group1']} vs {row['group2']}: "
#                       f"diff={row['meandiff']:.2f}, CI=[{row['lower']:.2f}, {row['upper']:.2f}], "
#                       f"p_bonf={row['p-adj']:.4f}{sig_marker}")
                
#         except Exception as e:
#             print(f"  ⚠ Post-hoc test failed: {e}")
    
#     # =============== SAVE RESULTS ===============
#     print(f"\n{'='*100}")
#     print(f"SAVING RESULTS")
#     print(f"{'='*100}")
    
#     # Save significant states as NPZ files
#     for feat in significant_results:
#         SEED = feat['seed']
#         K_CLUSTERS = feat['K']
#         best_cluster_id = feat['state']
#         cluster_labels_flat = feat['cluster_labels']
        
#         # Extract windows for significant cluster
#         cluster_windows_mask = cluster_labels_flat == best_cluster_id
#         significant_windows = X_scaled[cluster_windows_mask]
        
#         window_indices = np.where(cluster_windows_mask)[0]
#         subject_indices = window_indices // n_timepoints
#         timepoint_indices = window_indices % n_timepoints
#         window_labels = all_labels[subject_indices]
        
#         print(f"\nSeed {SEED}, K={K_CLUSTERS}, State {best_cluster_id}:")
#         print(f"  Windows: {len(significant_windows)}")
#         print(f"  Distribution: N={np.sum(window_labels == 0)}, "
#               f"A+P-={np.sum(window_labels == 1)}, A+P+={np.sum(window_labels == 2)}")
        
#         # Save NPZ
#         output_filename = OUTPUT_DIR / f"fMRI_K{K_CLUSTERS}_seed{SEED}_state{best_cluster_id}_FDR_significant.npz"
        
#         np.savez_compressed(
#             output_filename,
#             windows=significant_windows,
#             window_labels=window_labels,
#             subject_indices=subject_indices,
#             timepoint_indices=timepoint_indices,
#             cluster_id=best_cluster_id,
#             p_value_raw=feat['p_value'],
#             p_value_fdr_corrected=feat['p_value_fdr_corrected'],
#             test_type=feat['test'],
#             dominant_group=feat['dominant_group'],
#             seed=SEED,
#             k_clusters=K_CLUSTERS,
#             kmeans_centers=feat['kmeans_centers'],
#             all_cluster_labels=cluster_labels_flat,
#             # Metadata
#             win_step=WIN_STEP,
#             tr=TR,
#             n_subjects=n_subjects,
#             n_timepoints=n_timepoints,
#             n_features=n_features
#         )
#         print(f"  Saved: {output_filename}")

#     # Create summary DataFrame
#     summary_data = [{
#         'seed': f['seed'],
#         'k': f['K'],
#         'state': f['state'],
#         'p_value_raw': f['p_value'],
#         'p_value_fdr_corrected': f['p_value_fdr_corrected'],
#         'test_type': f['test'],
#         'dominant_group': f['dominant_group']
#     } for f in significant_results]
    
#     summary_df = pd.DataFrame(summary_data)
#     summary_csv = OUTPUT_DIR / "fMRI_significant_states_BH_FDR_corrected.csv"
#     summary_df.to_csv(summary_csv, index=False)
#     print(f"\n📁 Summary saved to: {summary_csv}")
    
#     # Save post-hoc results
#     if posthoc_results:
#         posthoc_df = pd.DataFrame(posthoc_results)
#         posthoc_csv = OUTPUT_DIR / "fMRI_posthoc_bonferroni_comparisons.csv"
#         posthoc_df.to_csv(posthoc_csv, index=False)
#         print(f"📁 Post-hoc results saved to: {posthoc_csv}")
        
#         # =============== GENERATE LATEX TABLE ===============
#         print(f"\n{'='*100}")
#         print("GENERATING LATEX TABLE FOR POST-HOC COMPARISONS")
#         print(f"{'='*100}")
        
#         # Filter to significant post-hoc results and sort by K
#         sig_posthoc = posthoc_df[posthoc_df['reject'] == True].sort_values(['K', 'state'])
        
#         # Generate LaTeX table
#         latex_output = OUTPUT_DIR / "posthoc_comparisons_table.tex"
        
#         with open(latex_output, 'w') as f:
#             f.write("\\begin{table}[t]\n")
#             f.write("\\centering\n")
#             f.write("\\footnotesize\n\n")
#             f.write("\\caption{Bonferroni-corrected post hoc comparisons of fMRI std-distance-to-centroid features across risk groups. ")
#             f.write("\\label{tab:postHocComparisons-fMRI}\n")
#             f.write("\\begin{tabular}{cllrrrrr}\n")
#             f.write("\\toprule\n")
#             f.write("\\textbf{State} & \\textbf{Group 1} & \\textbf{Group 2} & \\textbf{Diff.} & \n")
#             f.write("\\multicolumn{2}{c}{\\textbf{95\\% CI}} & \\textbf{df} & \\textbf{p$_{\\text{bonf}}$} \\\\\n")
#             f.write("\\cmidrule(lr){5-6}\n")
#             f.write(" & & & & \\textbf{Lower} & \\textbf{Upper} & & \\\\\n")
#             f.write("\\midrule\n\n")
            
#             # Group by K value
#             for k_val in sig_posthoc['K'].unique():
#                 k_data = sig_posthoc[sig_posthoc['K'] == k_val]
#                 n_rows = len(k_data)
                
#                 for idx, (_, row) in enumerate(k_data.iterrows()):
#                     if idx == 0:
#                         f.write(f"\\multirow{{{n_rows}}}{{*}}{{$K$ = {k_val}}} \n")
#                     else:
#                         f.write(" ")
                    
#                     # Format p-value significance
#                     p_val = row['p_adj']
#                     if p_val < 0.001:
#                         p_str = f"{p_val:.3f}$^*$"
#                     elif p_val < 0.01:
#                         p_str = f"{p_val:.3f}$^*$"
#                     elif p_val < POSTHOC_ALPHA:
#                         p_str = f"{p_val:.2f}$^*$"
#                     else:
#                         p_str = f"{p_val:.2f}"
                    
#                     # Format difference with sign
#                     diff = row['meandiff']
#                     diff_str = f"{diff:.2f}" if diff >= 0 else f"$-${abs(diff):.2f}"
                    
#                     # Format CI bounds
#                     lower = row['lower_ci']
#                     upper = row['upper_ci']
#                     lower_str = f"{lower:.2f}" if lower >= 0 else f"$-${abs(lower):.2f}"
#                     upper_str = f"{upper:.2f}" if upper >= 0 else f"$-${abs(upper):.2f}"
                    
#                     # Calculate degrees of freedom (n - k, where k = number of groups)
#                     df = n_subjects - 3  # 3 groups
                    
#                     f.write(f" & {row['group1']:<6} & {row['group2']:<6} & {diff_str:>8} & ")
#                     f.write(f"{lower_str:>7} & {upper_str:>7} & {df:>2} & {p_str} \\\\\n")
                
#                 # Add separator between K values
#                 if k_val != sig_posthoc['K'].max():
#                     f.write(" \n\\cmidrule[0.4pt]{1-8}\n\n")
            
#             f.write("\\bottomrule\n")
#             f.write("\\end{tabular}\n")
#             f.write("\\end{table}\n")
        
#         print(f"📄 LaTeX table saved to: {latex_output}")
#         print("\nPreview of LaTeX table:")
#         with open(latex_output, 'r') as f:
#             print(f.read())

# else:
#     print("No features found for correction!")

# print(f"\n{'='*100}")
# print("ANALYSIS COMPLETE WITH BENJAMINI-HOCHBERG FDR CORRECTION AND POST-HOC TESTING!")
# print(f"{'='*100}")

In [ ]:
#old

# import warnings
# import pandas as pd
# import numpy as np
# from pathlib import Path
# from scipy import stats
# from statsmodels.stats.multitest import multipletests
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
# from statsmodels.stats.multicomp import pairwise_tukeyhsd
# import os

# # Statistical tests (both parametric and non-parametric)
# from scipy.stats import kruskal, mannwhitneyu, shapiro, levene

# # ----------------- USER/PLOTTING CONFIG -----------------
# K_RANGE = range(5, 11)
# WIN_STEP = 5
# TR = 0.8
# SEEDS = [100, 200, 300, 400, 500]  # Five different seeds
# KMEANS_N_INIT = "auto"
# KMEANS_MAX_ITER = 10000
# PCA_RANDOM_STATE = 0
# P_VALUE_THRESHOLD = 0.05  # For FDR-BH correction
# POSTHOC_ALPHA = 0.05

# # Statistical test parameters
# NORMALITY_ALPHA = 0.05
# VARIANCE_ALPHA = 0.05
# MIN_SAMPLE_SIZE = 3

# # Create output directory
# OUTPUT_DIR = Path("fMRI_windows_analysis")
# OUTPUT_DIR.mkdir(exist_ok=True)

# # ---- Sanity checks ----
# if "all_data" not in globals() or "all_labels" not in globals():
#     raise RuntimeError("Please define `all_data` and `all_labels` before running.")
# if np.isnan(all_data).any():
#     raise ValueError("all_data contains NaNs. Please clean or impute before running.")

# n_subjects, n_timepoints, n_features = all_data.shape
# print(f"Data: {n_subjects} subjects × {n_timepoints} timepoints × {n_features} features")
# print("Using statistical tests with normality and variance checks + FDR-BH correction")

# # ---------------- PREP DATA ----------------
# reshaped_data = all_data.reshape(-1, n_features)
# X_scaled = reshaped_data
# print(f"Reshaped data for clustering / PCA: {reshaped_data.shape}")

# def test_normality_and_variance(groups, state_id, k_clusters):
#     """Test normality and homogeneity of variance for groups."""
#     test_details = {
#         'state': state_id,
#         'k': k_clusters,
#         'n_groups': len(groups),
#         'group_sizes': [len(g) for g in groups],
#         'normality_test': 'Not performed',
#         'variance_test': 'Not performed',
#         'use_parametric': False,
#         'reason': ''
#     }
    
#     if any(len(g) < MIN_SAMPLE_SIZE for g in groups):
#         test_details['reason'] = f'Sample size too small (min {MIN_SAMPLE_SIZE} required)'
#         return False, test_details
    
#     # Test normality using Shapiro-Wilk
#     normality_pvals = []
#     for group in groups:
#         if len(group) >= 3:
#             try:
#                 _, p_val = shapiro(group)
#                 normality_pvals.append(p_val)
#             except:
#                 normality_pvals.append(0.0)
#         else:
#             normality_pvals.append(0.0)
    
#     all_normal = all(p > NORMALITY_ALPHA for p in normality_pvals)
#     test_details['normality_test'] = f"Shapiro-Wilk p-values: {[f'{p:.4f}' for p in normality_pvals]}"
    
#     if not all_normal:
#         test_details['reason'] = 'One or more groups not normally distributed'
#         return False, test_details
    
#     # Test variance homogeneity using Levene's test
#     try:
#         _, variance_p = levene(*groups)
#         test_details['variance_test'] = f"Levene's test p-value: {variance_p:.4f}"
        
#         if variance_p <= VARIANCE_ALPHA:
#             test_details['reason'] = 'Unequal variances detected'
#             return False, test_details
#     except:
#         test_details['variance_test'] = 'Levene test failed'
#         test_details['reason'] = 'Could not test variance homogeneity'
#         return False, test_details
    
#     test_details['reason'] = 'All parametric assumptions satisfied'
#     test_details['use_parametric'] = True
#     return True, test_details

# def choose_statistical_test(groups, state_id, k_clusters):
#     """Choose appropriate statistical test based on assumptions."""
#     # Basic checks
#     if len(groups) < 2 or any(g.size == 0 for g in groups):
#         test_details = {
#             'state': state_id, 'k': k_clusters, 'test_used': 'None',
#             'reason': 'Insufficient groups or empty groups'
#         }
#         return 'None', 1.0, 0.0, test_details
    
#     if np.std(np.concatenate(groups)) <= 1e-10:
#         test_details = {
#             'state': state_id, 'k': k_clusters, 'test_used': 'None',
#             'reason': 'No variance in data'
#         }
#         return 'None', 1.0, 0.0, test_details
    
#     group_stds = [np.std(g) for g in groups]
#     if any(std <= 1e-10 for std in group_stds):
#         test_details = {
#             'state': state_id, 'k': k_clusters, 'test_used': 'None',
#             'reason': 'One or more groups has no variance'
#         }
#         return 'None', 1.0, 0.0, test_details
    
#     # Test assumptions
#     use_parametric, test_details = test_normality_and_variance(groups, state_id, k_clusters)
    
#     try:
#         if use_parametric:
#             f_stat, p_value = stats.f_oneway(*groups)
#             test_details['test_used'] = 'ANOVA' if len(groups) > 2 else 'F-test'
#             return test_details['test_used'], p_value, f_stat, test_details
#         else:
#             if len(groups) == 2:
#                 statistic, p_value = mannwhitneyu(groups[0], groups[1], alternative='two-sided')
#                 test_details['test_used'] = 'Mann-Whitney'
#                 return 'Mann-Whitney', p_value, statistic, test_details
#             else:
#                 statistic, p_value = kruskal(*groups)
#                 test_details['test_used'] = 'Kruskal-Wallis'
#                 return 'Kruskal-Wallis', p_value, statistic, test_details
#     except Exception as e:
#         test_details['reason'] = f'Statistical test failed: {str(e)}'
#         test_details['test_used'] = 'None'
#         return 'None', 1.0, 0.0, test_details

# def compute_std_distance_to_centroid(cluster_labels_flat, X_data, kmeans_model, n_subjects, n_timepoints):
#     """Compute std_distance_to_centroid for each subject and state."""
#     k_clusters = kmeans_model.n_clusters
#     std_distances = np.full((n_subjects, k_clusters), np.nan)
    
#     for s in range(n_subjects):
#         for k in range(k_clusters):
#             # Get windows for this subject in this cluster
#             mask = (cluster_labels_flat == k) & (np.arange(len(cluster_labels_flat)) // n_timepoints == s)
#             cluster_points = X_data[mask]
            
#             if cluster_points.shape[0] > 0:
#                 # Compute distances to centroid
#                 distances = np.linalg.norm(cluster_points - kmeans_model.cluster_centers_[k], axis=1)
#                 std_distances[s, k] = np.std(distances)
    
#     return std_distances

# # Store ALL results for FDR correction
# all_significant_features = []

# # ---- Loop through all seeds ----
# for SEED in SEEDS:
#     print(f"\n{'='*100}")
#     print(f"PROCESSING SEED {SEED}")
#     print(f"{'='*100}")
    
#     # ---- Loop through all K values for this seed ----
#     for K_CLUSTERS in K_RANGE:
#         print(f"\n{'-'*60}")
#         print(f"SEED {SEED}: RUNNING ANALYSIS FOR K = {K_CLUSTERS}")
#         print(f"{'-'*60}")

#         # ---------------- KMEANS ----------------
#         kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=SEED, 
#                        n_init=KMEANS_N_INIT, max_iter=KMEANS_MAX_ITER)
#         cluster_labels_flat = kmeans.fit_predict(X_scaled)
#         cluster_labels = cluster_labels_flat.reshape(n_subjects, n_timepoints)

#         # Compute std_distance_to_centroid features
#         all_std_distances = compute_std_distance_to_centroid(
#             cluster_labels_flat, X_scaled, kmeans, n_subjects, n_timepoints
#         )
        
#         # ----------------- STATISTICAL TESTING for each state -----------------
#         unique_labels = np.unique(all_labels)
        
#         for state in range(K_CLUSTERS):
#             vals = all_std_distances[:, state]
            
#             # Handle NaN values
#             valid_mask = ~np.isnan(vals)
#             if not np.any(valid_mask):
#                 continue
            
#             valid_vals = vals[valid_mask]
#             valid_labels = all_labels[valid_mask]
            
#             if len(np.unique(valid_labels)) < 2:
#                 continue
            
#             # Create groups for statistical testing
#             groups = [valid_vals[valid_labels == label] for label in unique_labels]
#             groups = [g for g in groups if g.size > 0]
            
#             # Statistical test
#             test_type, p_value, f_stat, test_details = choose_statistical_test(groups, state, K_CLUSTERS)
            
#             if test_type != 'None':
#                 # Calculate group means
#                 group_means = {label: np.mean(valid_vals[valid_labels == label]) 
#                               for label in unique_labels if np.any(valid_labels == label)}
                
#                 # Determine dominant group
#                 dominant_label = max(group_means, key=group_means.get)
#                 label_to_group = {0: 'N', 1: 'A+P-', 2: 'A+P+'}
#                 dominant_group = label_to_group.get(dominant_label, f'Group{dominant_label}')
                
#                 # Store result (before correction)
#                 all_significant_features.append({
#                     'seed': SEED,
#                     'K': K_CLUSTERS,
#                     'feature_type': 'std_distance_to_centroid',
#                     'state': state,
#                     'p_value': p_value,
#                     'test': test_type,
#                     'dominant_group': dominant_group,
#                     'group_means': group_means,
#                     'feature_values': vals,  # Include NaN values
#                     'cluster_labels': cluster_labels_flat,
#                     'kmeans_centers': kmeans.cluster_centers_
#                 })
                
#                 print(f"  State {state}: {test_type} p={p_value:.6f} ({dominant_group} higher)")

# # =============== FDR-BH CORRECTION (Like EEG code) ===============
# print(f"\n{'='*100}")
# print("APPLYING FDR-BH CORRECTION")
# print(f"{'='*100}")

# if all_significant_features:
#     p_values = [feat['p_value'] for feat in all_significant_features]
#     reject, pvals_corrected, _, _ = multipletests(p_values, alpha=P_VALUE_THRESHOLD, method='fdr_bh')
    
#     for i, feat in enumerate(all_significant_features):
#         feat['p_value_corrected'] = pvals_corrected[i]
#         feat['significant_after_correction'] = reject[i]
    
#     print("\nCorrected p-values using FDR-BH:")
#     for feat in all_significant_features:
#         if feat['significant_after_correction']:
#             print(f" ✓ Seed={feat['seed']}, K={feat['K']}, State {feat['state']}: "
#                   f"corrected p={feat['p_value_corrected']:.6f}")

#     # Filter to significant results only
#     significant_results_corrected = [f for f in all_significant_features if f['significant_after_correction']]
    
#     print(f"\n{'='*100}")
#     print(f"FINAL RESULTS: {len(significant_results_corrected)} significant states after FDR-BH correction")
#     print(f"{'='*100}")
    
#     # Save ONLY significant states
#     for feat in significant_results_corrected:
#         SEED = feat['seed']
#         K_CLUSTERS = feat['K']
#         best_cluster_id = feat['state']
#         cluster_labels_flat = feat['cluster_labels']
        
#         # Extract windows for significant cluster
#         cluster_windows_mask = cluster_labels_flat == best_cluster_id
#         significant_windows = X_scaled[cluster_windows_mask]
        
#         window_indices = np.where(cluster_windows_mask)[0]
#         subject_indices = window_indices // n_timepoints
#         timepoint_indices = window_indices % n_timepoints
#         window_labels = all_labels[subject_indices]
        
#         print(f"\nSeed {SEED}, K={K_CLUSTERS}, State {best_cluster_id}:")
#         print(f"  Windows: {len(significant_windows)}")
#         print(f"  Distribution: N={np.sum(window_labels == 0)}, "
#               f"A+P-={np.sum(window_labels == 1)}, A+P+={np.sum(window_labels == 2)}")
        
#         # Save NPZ
#         output_filename = OUTPUT_DIR / f"fMRI_K{K_CLUSTERS}_seed{SEED}_state{best_cluster_id}_significant.npz"
        
#         np.savez_compressed(
#             output_filename,
#             windows=significant_windows,
#             window_labels=window_labels,
#             subject_indices=subject_indices,
#             timepoint_indices=timepoint_indices,
#             cluster_id=best_cluster_id,
#             p_value_uncorrected=feat['p_value'],
#             p_value_corrected=feat['p_value_corrected'],
#             test_type=feat['test'],
#             dominant_group=feat['dominant_group'],
#             seed=SEED,
#             k_clusters=K_CLUSTERS,
#             kmeans_centers=feat['kmeans_centers'],
#             all_cluster_labels=cluster_labels_flat,
#             # Metadata
#             win_step=WIN_STEP,
#             tr=TR,
#             n_subjects=n_subjects,
#             n_timepoints=n_timepoints,
#             n_features=n_features
#         )
#         print(f"  Saved: {output_filename}")

#     # Create summary DataFrame
#     summary_data = [{
#         'seed': f['seed'],
#         'k': f['K'],
#         'state': f['state'],
#         'p_value_uncorrected': f['p_value'],
#         'p_value_corrected': f['p_value_corrected'],
#         'test_type': f['test'],
#         'dominant_group': f['dominant_group']
#     } for f in significant_results_corrected]
    
#     summary_df = pd.DataFrame(summary_data)
#     summary_csv = OUTPUT_DIR / "fMRI_significant_states_FDR_corrected.csv"
#     summary_df.to_csv(summary_csv, index=False)
#     print(f"\nSummary saved to: {summary_csv}")

# else:
#     print("No features found for correction!")

# print(f"\n{'='*100}")
# print("ANALYSIS COMPLETE WITH FDR-BH CORRECTION!")
# print(f"{'='*100}")

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
import templateflow.api as tf
import nibabel as nib
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. SETUP AND LOAD ATLAS ---
print("Loading atlas information...")
try:
    # Load Schaefer atlas TSV for region names and network info
    atlas_tsv_path = tf.get(
        template='MNI152NLin2009cAsym', atlas='Schaefer2018',
        desc='100Parcels7Networks', extension='.tsv'
    )
    # The first column is the index, which corresponds to the ROI label (1-100)
    atlas_df = pd.read_csv(atlas_tsv_path, sep='\t', index_col=0)
    atlas_loaded = True
    print("Atlas information loaded successfully.")
except Exception as e:
    print(f"Error loading atlas information: {e}")
    atlas_df = None
    atlas_loaded = False

# Dictionary to map network abbreviations to full names
NETWORK_MAP = {
    "Vis": "Visual", "SomMot": "Somatomotor", "DorsAttn": "Dorsal Attention",
    "SalVentAttn": "Salience/Ventral Attention", "Limbic": "Limbic",
    "Cont": "Control", "Default": "Default Mode"
}

# --- 2. HELPER FUNCTIONS (WITH CORRECTIONS) ---

def lower_triangle_to_indices(k, n=100):
    """
    Convert a flattened lower triangle index to (i,j) matrix indices.
    For a matrix of size n×n, the lower triangle has n(n-1)/2 elements.
    
    Parameters:
    - k: Index in the flattened lower triangle (0-based)
    - n: Size of the square matrix
    
    Returns:
    - (i,j): 0-based indices in the square matrix
    """
    # Formula for converting lower triangular index to (i,j) coordinates
    # This is the correct mathematical formula for this conversion
    i = int(n - 2 - np.floor(np.sqrt(-8*k + 4*n*(n-1)-7)/2 - 0.5))
    j = int(k + i + 1 - n*(n-1)/2 + (n-i)*((n-i)-1)/2)
    return i, j

def get_roi_info_detailed(roi_idx):
    """
    Get detailed information about a region of interest from the atlas.
    Handles 0-based ROI indices (0-99) for the 100-region atlas.
    """
    # Check if index is valid (should be 0-99 for 100-region atlas)
    if not atlas_loaded or roi_idx < 0 or roi_idx >= 100:
        return {
            'hemisphere': 'N/A',
            'network_full': 'Unknown', 
            'region_full': f'ROI_{roi_idx}',
            'full_name': f'ROI_{roi_idx}'
        }
    
    # Atlas is 1-indexed, so add 1 to the 0-based roi_idx for lookup
    roi_label = roi_idx + 1
    
    # Get the name from the atlas dataframe
    original_name = atlas_df.loc[roi_label, 'name']
    
    # Parse the name components
    name_parts = original_name.replace('7Networks_', '').split('_')
    hemisphere = name_parts[0]
    network_abbr = name_parts[1]
    region_parts = name_parts[2:]
    
    network_full = NETWORK_MAP.get(network_abbr, network_abbr)
    region_full = "_".join(region_parts)
    
    # Create a readable full name
    full_name = f"{hemisphere} {network_full} - {region_full}"

    return {
        'hemisphere': hemisphere,
        'network_full': network_full,
        'region_full': region_full,
        'full_name': full_name
    }
def get_top_connections_per_file(windows, top_n=10, n_regions=100):
    """Finds the top N connections from a set of correlation windows."""
    # Use median for robustness against outliers
    mean_correlations = np.median(windows, axis=0)
    
    # Get indices of top N values in descending order
    top_indices = np.argsort(mean_correlations)[-top_n:][::-1]
    top_values = mean_correlations[top_indices]
    
    top_pairs = []
    for idx, value in zip(top_indices, top_values):
        i, j = lower_triangle_to_indices(idx, n_regions)
        # Standardize pair order (smaller_index, larger_index)
        pair = (min(i, j), max(i, j))
        top_pairs.append({'pair': pair, 'correlation': value, 'flat_index': idx})
    
    return top_pairs

# --- 3. ANALYSIS PIPELINE ---

# Load all saved .npz files
OUTPUT_DIR = Path("fMRI_windows_analysis")
loaded_files = list(OUTPUT_DIR.glob("*.npz"))
print(f"\nFound {len(loaded_files)} files in {OUTPUT_DIR} for robust analysis.")

# Data stores for aggregation
all_top_pairs_data = []
pair_frequency = Counter()
pair_correlations = {}

print(f"\n{'='*120}\nANALYZING TOP 10 CONNECTIONS PER FILE\n{'='*120}")

# Process each file to find its top connections
for file_idx, file_path in enumerate(loaded_files):
    data = np.load(file_path)
    windows = data['windows']
    
    top_connections = get_top_connections_per_file(windows, top_n=10, n_regions=100)
    
    print(f"\nFile {file_idx+1}/{len(loaded_files)}: {file_path.name} ({windows.shape[0]} windows)")
    
    for rank, conn_info in enumerate(top_connections, 1):
        pair = conn_info['pair']
        correlation = conn_info['correlation']
        
        # Aggregate frequency and correlation values for each pair
        pair_frequency[pair] += 1
        if pair not in pair_correlations:
            pair_correlations[pair] = []
        pair_correlations[pair].append(correlation)
        
        # Store detailed info for later
        all_top_pairs_data.append({
            'file_idx': file_idx, 'rank': rank, 'pair': pair, 'correlation': correlation
        })
        
        # Print info for this connection
        roi_i_info = get_roi_info_detailed(pair[0])
        roi_j_info = get_roi_info_detailed(pair[1])
        print(f"  {rank:2d}. Pair {pair}: {correlation:.4f} - {roi_i_info['region_full']} ↔ {roi_j_info['region_full']}")

# --- 4. CONSISTENCY ANALYSIS ---
print(f"\n\n{'='*120}\nCONSISTENCY ANALYSIS ACROSS ALL FILES\n{'='*120}")

# Calculate statistics for every pair found
pair_stats = []
for pair, frequency in pair_frequency.items():
    correlations = pair_correlations[pair]
    i, j = pair
    roi_i_info = get_roi_info_detailed(i)
    roi_j_info = get_roi_info_detailed(j)
    
    pair_stats.append({
        'pair': pair,
        'region_i_idx': i, 'region_j_idx': j,
        'region_i_hemisphere': roi_i_info['hemisphere'], 'region_i_network': roi_i_info['network_full'], 'region_i_name': roi_i_info['region_full'],
        'region_j_hemisphere': roi_j_info['hemisphere'], 'region_j_network': roi_j_info['network_full'], 'region_j_name': roi_j_info['region_full'],
        'frequency': frequency,
        'frequency_percent': (frequency / len(loaded_files)) * 100,
        'mean_correlation': np.mean(correlations),
        'std_correlation': np.std(correlations),
        'connection_description': f"{roi_i_info['full_name']} ↔ {roi_j_info['full_name']}"
    })

# Create a DataFrame and sort by frequency, then correlation
pair_stats_df = pd.DataFrame(pair_stats).sort_values(['frequency', 'mean_correlation'], ascending=[False, False])

# --- 5. DISPLAY AND SAVE RESULTS ---

# Display the most consistent connections
print(f"\nMOST CONSISTENT BRAIN REGION PAIRS (Top 20):")
print(f"{'Rank':<5} {'Pair':<10} {'Freq':<8} {'%':<6} {'Mean±Std':<15} {'Connection'}")
print(f"{'-'*150}")

for rank, row in pair_stats_df.head(20).iterrows():
    freq_str = f"{row['frequency']}/{len(loaded_files)}"
    percent_str = f"{row['frequency_percent']:.1f}%"
    corr_str = f"{row['mean_correlation']:.3f}±{row['std_correlation']:.3f}"
    
    print(f"{rank+1:<5} {str(row['pair']):<10} {freq_str:<8} {percent_str:<6} {corr_str:<15} {row['connection_description']}")

# Save detailed results to CSV files
output_csv = OUTPUT_DIR / "robust_brain_connections_consistency_analysis.csv"
pair_stats_df.to_csv(output_csv, index=False)
print(f"\nDetailed consistency analysis saved to: {output_csv}")

all_pairs_df = pd.DataFrame(all_top_pairs_data)
per_file_csv = OUTPUT_DIR / "robust_brain_connections_per_file.csv"
all_pairs_df.to_csv(per_file_csv, index=False)
print(f"Per-file detailed results saved to: {per_file_csv}")

# --- 6. SUMMARY STATISTICS ---
print(f"\n\n{'='*120}\nROBUST ANALYSIS SUMMARY\n{'='*120}")
print(f"📊 FILES ANALYZED: {len(loaded_files)}")
print(f"📊 UNIQUE BRAIN REGION PAIRS FOUND: {len(pair_stats_df)}")
print(f"📊 PAIRS APPEARING IN ALL FILES: {len(pair_stats_df[pair_stats_df['frequency'] == len(loaded_files)])}")
print(f"📊 PAIRS APPEARING IN ≥75% FILES: {len(pair_stats_df[pair_stats_df['frequency'] >= 0.75 * len(loaded_files)])}")
print(f"📊 PAIRS APPEARING IN ≥50% FILES: {len(pair_stats_df[pair_stats_df['frequency'] >= 0.5 * len(loaded_files)])}")

print(f"\n✅ Robust consistency analysis complete!")


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import templateflow.api as tf
import nibabel as nib
from nilearn import image
from scipy import ndimage
from nilearn.plotting import plot_connectome, view_connectome
from collections import Counter
from pathlib import Path

# First, let's extract the Schaefer atlas coordinates
print("Loading Schaefer atlas...")

# Load the atlas file
atlas_paths = tf.get(
    template='MNI152NLin2009cAsym', atlas='Schaefer2018',
    desc='100Parcels7Networks', extension='.nii.gz'
)

atlas_path = atlas_paths[0] if isinstance(atlas_paths, list) else atlas_paths
atlas_img = nib.load(atlas_path)
atlas_data = atlas_img.get_fdata()

def get_region_coordinates(atlas_data, atlas_img):
    """Extract center of mass coordinates for each region"""
    coordinates = []
    region_labels = []
    
    # Get unique labels (excluding background = 0)
    unique_labels = np.unique(atlas_data)[1:]  # Skip 0 (background)
    
    for label in unique_labels:
        # Create mask for this region
        region_mask = (atlas_data == label)
        
        # Calculate center of mass in voxel coordinates
        com_voxel = ndimage.center_of_mass(region_mask)
        
        # Convert voxel coordinates to MNI coordinates
        com_mni = image.coord_transform(
            com_voxel[0], com_voxel[1], com_voxel[2], 
            atlas_img.affine
        )
        
        coordinates.append([com_mni[0], com_mni[1], com_mni[2]])
        region_labels.append(int(label))
    
    return np.array(coordinates), region_labels

# Extract coordinates
coordinates, region_labels = get_region_coordinates(atlas_data, atlas_img)

# Also load the TSV file for region names
atlas_tsv_paths = tf.get(
    template='MNI152NLin2009cAsym', atlas='Schaefer2018',
    desc='100Parcels7Networks', extension='.tsv'
)
atlas_tsv_path = atlas_tsv_paths[0] if isinstance(atlas_tsv_paths, list) else atlas_tsv_paths
# The Schaefer atlas TSV has 1-indexed ROI labels (1-100) as the index column
atlas_df = pd.read_csv(atlas_tsv_path, sep='\t', index_col=0)

# Extract region names and network information with hemisphere
roi_names = [name.replace('7Networks_', '') for name in atlas_df['name']]
# Modified to include hemisphere in the short name
roi_names_short = [name.split('_')[0] + '_' + name.split('_')[2] + '_' + name.split('_')[1] for name in roi_names]
networks = [name.split("_")[1] for name in roi_names]
hemispheres = [name.split("_")[0] for name in roi_names]

print(f"✅ Loaded {len(coordinates)} region coordinates")

# Create regions info dataframe
regions_info = pd.DataFrame({
    'region_id': range(len(coordinates)),
    'region_name': roi_names_short,
    'network': networks,
    'hemisphere': hemispheres,
    'full_name': roi_names,
    'x_mni': coordinates[:, 0],
    'y_mni': coordinates[:, 1],
    'z_mni': coordinates[:, 2]
})

# Dictionary to map network abbreviations to full names
NETWORK_MAP = {
    "Vis": "Visual",
    "SomMot": "Somatomotor",
    "DorsAttn": "Dorsal Attention",
    "SalVentAttn": "Salience/Ventral Attention",
    "Limbic": "Limbic",
    "Cont": "Control",
    "Default": "Default Mode"
}

def get_roi_info_detailed(roi_idx):
    """
    Get detailed information about a region of interest from the atlas.
    Handles 0-based ROI indices (0-99) for the 100-region atlas.
    """
    # Check if index is valid (should be 0-99 for 100-region atlas)
    if not atlas_df is None and 0 <= roi_idx < 100:
        # Convert to 1-based indexing for atlas lookup (atlas is 1-100)
        roi_label = int(roi_idx) + 1
        
        # Get the name from the atlas dataframe
        original_name = atlas_df.loc[roi_label, 'name']
        
        # Parse the name components
        name_parts = original_name.replace('7Networks_', '').split('_')
        hemisphere = name_parts[0]
        network_abbr = name_parts[1]
        region_parts = name_parts[2:]
        
        network_full = NETWORK_MAP.get(network_abbr, network_abbr)
        region_full = "_".join(region_parts)
        
        # Create a readable full name
        full_name = f"{hemisphere} {network_full} - {region_full}"

        return {
            'hemisphere': hemisphere,
            'network_full': network_full,
            'region_full': region_full,
            'full_name': full_name
        }
    else:
        # Return placeholder for invalid indices
        return {
            'hemisphere': 'N/A',
            'network_full': 'Unknown',
            'region_full': f'ROI_{roi_idx}',
            'full_name': f'ROI_{roi_idx}'
        }

def lower_triangle_to_indices(k, n=100):
    """
    Convert a flattened lower triangle index to (i,j) matrix indices.
    For a matrix of size n×n, the lower triangle has n(n-1)/2 elements.
    
    Parameters:
    - k: Index in the flattened lower triangle (0-based)
    - n: Size of the square matrix
    
    Returns:
    - (i,j): 0-based indices in the square matrix
    """
    # Formula for converting lower triangular index to (i,j) coordinates
    # This is the correct mathematical formula for this conversion
    i = int(n - 2 - np.floor(np.sqrt(-8*k + 4*n*(n-1)-7)/2 - 0.5))
    j = int(k + i + 1 - n*(n-1)/2 + (n-i)*((n-i)-1)/2)
    return i, j

def get_top_connections_per_file(windows, top_n=10, n_regions=100):
    """Finds the top N connections from a set of correlation windows."""
    # Use median for robustness against outliers
    mean_correlations = np.median(windows, axis=0)
    
    # Get indices of top N values in descending order
    top_indices = np.argsort(mean_correlations)[-top_n:][::-1]
    top_values = mean_correlations[top_indices]
    
    top_pairs = []
    for idx, value in zip(top_indices, top_values):
        i, j = lower_triangle_to_indices(idx, n_regions)
        # Standardize pair order (smaller_index, larger_index)
        pair = (min(i, j), max(i, j))
        top_pairs.append({'pair': pair, 'correlation': value, 'flat_index': idx})
    
    return top_pairs

# Now load your correlation results and create adjacency matrix
print("\n" + "="*80)
print("CREATING ROBUST CORRELATION-BASED CONNECTOME VISUALIZATION")
print("="*80)

# Load all files and extract windows with robust consistency analysis
OUTPUT_DIR = Path("fMRI_windows_analysis")
loaded_files = list(OUTPUT_DIR.glob("*.npz"))
print(f"\nFound {len(loaded_files)} files for robust connectome analysis.")

# Store results for each file
file_results = []
pair_frequency = Counter()
pair_correlations = {}

print(f"\n{'='*120}")
print("ANALYZING TOP 10 CONNECTIONS PER FILE FOR CONNECTOME")
print(f"{'='*120}")

# Process each file individually
for file_idx, file_path in enumerate(loaded_files):
    data = np.load(file_path)
    windows = data['windows']
    
    # Get file metadata
    seed = data['seed'] if 'seed' in data else 'unknown'
    k_clusters = data['k_clusters'] if 'k_clusters' in data else 'unknown'
    
    print(f"File {file_idx+1}/{len(loaded_files)}: {file_path.name}")
    print(f"  Seed: {seed}, K: {k_clusters}")
    print(f"  Windows shape: {windows.shape}")
    
    # Get top 10 connections for this file
    top_connections = get_top_connections_per_file(windows, top_n=10, n_regions=100)
    
    # Track pair frequencies and correlations
    for rank, conn_info in enumerate(top_connections, 1):
        pair = conn_info['pair']
        correlation = conn_info['correlation']
        flat_index = conn_info['flat_index']
        
        i, j = int(pair[0]), int(pair[1])  # Ensure integers
        pair = (i, j)  # Recreate as integers
        
        # Count frequency
        pair_frequency[pair] += 1
        
        # Store correlations for this pair
        if pair not in pair_correlations:
            pair_correlations[pair] = []
        pair_correlations[pair].append(correlation)
        
        # For debugging, print the actual indices and region names
        roi_i_info = get_roi_info_detailed(i)
        roi_j_info = get_roi_info_detailed(j)
        print(f"  {rank:2d}. Pair ({i:2d},{j:2d}): {correlation:.4f} - {roi_i_info['region_full']} ↔ {roi_j_info['region_full']}")

# Calculate statistics for each pair
pair_stats = []
for pair, frequency in pair_frequency.items():
    correlations = pair_correlations[pair]
    
    i, j = int(pair[0]), int(pair[1])  # Ensure integers
    roi_i_info = get_roi_info_detailed(i)
    roi_j_info = get_roi_info_detailed(j)
    
    stats_dict = {
        'pair': (i, j),  # Store as integers
        'region_i_idx': i,
        'region_j_idx': j,
        'region_i_hemisphere': roi_i_info['hemisphere'],
        'region_i_network': roi_i_info['network_full'],
        'region_i_name': roi_i_info['region_full'],
        'region_j_hemisphere': roi_j_info['hemisphere'],
        'region_j_network': roi_j_info['network_full'],
        'region_j_name': roi_j_info['region_full'],
        'frequency': frequency,
        'frequency_percent': (frequency / len(loaded_files)) * 100,
        'mean_correlation': np.mean(correlations),
        'std_correlation': np.std(correlations),
        'min_correlation': np.min(correlations),
        'max_correlation': np.max(correlations),
        'connection_description': f"{roi_i_info['full_name']} ↔ {roi_j_info['full_name']}"
    }
    pair_stats.append(stats_dict)

# Sort by frequency (most consistent first), then by mean correlation
pair_stats_df = pd.DataFrame(pair_stats)
pair_stats_df = pair_stats_df.sort_values(['frequency', 'mean_correlation'], ascending=[False, False])

# Use top 10 most consistent connections for visualization
top_consistent = pair_stats_df.head(10)

print(f"\nTOP 10 MOST CONSISTENT BRAIN REGION PAIRS FOR CONNECTOME:")
print(f"{'Rank':<4} {'ROI Pair':<12} {'Mean±Std':<15} {'Freq':<8} {'Connection':<70} {'Networks'}")
print("-" * 140)

# Create adjacency matrix for top consistent connections
n_rois = 100
adj_matrix = np.zeros((n_rois, n_rois))
top_connections = []

for rank, (_, row) in enumerate(top_consistent.iterrows(), 1):
    i, j = int(row['region_i_idx']), int(row['region_j_idx'])  # Ensure integers
    mean_corr = row['mean_correlation']
    std_corr = row['std_correlation']
    frequency = row['frequency']
    
    # Set adjacency matrix values (symmetric)
    adj_matrix[i, j] = mean_corr
    adj_matrix[j, i] = mean_corr
    
    # Get region information from regions_info with hemisphere
    region_i_name = regions_info.iloc[i]['region_name']  # Already includes hemisphere
    region_j_name = regions_info.iloc[j]['region_name']  # Already includes hemisphere
    network_i = regions_info.iloc[i]['network']
    network_j = regions_info.iloc[j]['network']
    hemisphere_i = regions_info.iloc[i]['hemisphere']
    hemisphere_j = regions_info.iloc[j]['hemisphere']
    
    # Create connection description with hemisphere information
    # Extract just the region name part (without hemisphere for cleaner display)
    region_i_clean = "_".join(region_i_name.split('_')[1:])  # Remove hemisphere prefix
    region_j_clean = "_".join(region_j_name.split('_')[1:])  # Remove hemisphere prefix
    
    connection_desc = f"{hemisphere_i} {region_i_clean} ↔ {hemisphere_j} {region_j_clean}"
    network_desc = f"{network_i} ↔ {network_j}"
    corr_desc = f"{mean_corr:.3f}±{std_corr:.3f}"
    freq_desc = f"{frequency}/{len(loaded_files)}"
    
    print(f"{rank:<4} ({i:2d},{j:2d})      {corr_desc:<15} {freq_desc:<8} {connection_desc:<70} {network_desc}")
    
    top_connections.append({
        'rank': rank,
        'roi_i': i,
        'roi_j': j,
        'mean_correlation': mean_corr,
        'std_correlation': std_corr,
        'frequency': frequency,
        'frequency_percent': row['frequency_percent'],
        'region_i': region_i_name,  # Full name with hemisphere
        'region_j': region_j_name,  # Full name with hemisphere
        'hemisphere_i': hemisphere_i,
        'hemisphere_j': hemisphere_j,
        'region_i_clean': region_i_clean,
        'region_j_clean': region_j_clean,
        'network_i': network_i,
        'network_j': network_j,
        'region_i_full': row['region_i_name'],
        'region_j_full': row['region_j_name'],
        'connection_with_hemisphere': connection_desc
    })

print(f"\n✅ Created adjacency matrix with {np.count_nonzero(adj_matrix)} most consistent connections")

# Save adjacency matrix
node_names = [f"ROI_{i}_{regions_info.iloc[i]['region_name']}" for i in range(n_rois)]
adj_df = pd.DataFrame(adj_matrix, index=node_names, columns=node_names)
output_adj_csv = OUTPUT_DIR / "robust_consistent_correlations_adjacency_matrix.csv"
adj_df.to_csv(output_adj_csv, index=True)
print(f"📁 Saved adjacency matrix to: {output_adj_csv}")

# Create detailed connections DataFrame
connections_df = pd.DataFrame(top_connections)
output_connections_csv = OUTPUT_DIR / "robust_consistent_correlations_detailed.csv"
connections_df.to_csv(output_connections_csv, index=False)
print(f"📁 Saved detailed connections to: {output_connections_csv}")

# Save full consistency analysis
output_full_csv = OUTPUT_DIR / "robust_connectome_consistency_analysis.csv"
pair_stats_df.to_csv(output_full_csv, index=False)
print(f"📁 Saved full consistency analysis to: {output_full_csv}")

# Visualization 1: Static brain connectome plot
print("\n🧠 Creating robust static brain connectome visualization...")

plt.figure(figsize=(15, 10))
plot_connectome(
    adj_matrix, 
    coordinates,
    node_color='orange',
    edge_cmap='viridis',
    title=f'Top 10 Most Consistent Brain Region Correlations\n(Across {len(loaded_files)} significant cluster analyses)',
    annotate=True,
    edge_threshold='95%'  # Only show strongest connections
)

output_plot = OUTPUT_DIR / "robust_consistent_correlations_brain_connectome.png"
plt.savefig(output_plot, dpi=300, bbox_inches='tight')
plt.show()
print(f"📊 Saved static connectome plot to: {output_plot}")

# Visualization 3: Interactive brain connectome
print("\n🌐 Creating robust interactive brain connectome...")

# Filter adjacency matrix to show only consistent connections
adj_filtered = adj_matrix.copy()

interactive_view = view_connectome(
    adj_filtered, 
    coordinates, 
    node_size=5,
    node_color="gray",
    edge_cmap='Blues',
    colorbar=False,
)

# Save interactive view
output_interactive = OUTPUT_DIR / "robust_consistent_correlations_interactive.html"
interactive_view.save_as_html(str(output_interactive))
print(f"🌐 Saved interactive view to: {output_interactive}")

# Enhanced Network analysis summary with hemisphere information
print(f"\n" + "="*80)
print("ROBUST NETWORK CONNECTIVITY SUMMARY")
print("="*80)

network_connections = {}
within_network = 0
between_network = 0
consistency_analysis = []

for conn in top_connections:
    net_i = conn['network_i']
    net_j = conn['network_j']
    
    if net_i == net_j:
        within_network += 1
        conn_type = "Within-network"
    else:
        between_network += 1
        conn_type = "Between-network"
    
    net_pair = f"{min(net_i, net_j)} ↔ {max(net_i, net_j)}"
    if net_pair not in network_connections:
        network_connections[net_pair] = []
    network_connections[net_pair].append(conn)
    
    print(f"Rank {conn['rank']}: {conn_type} ({net_pair})")
    print(f"         {conn['connection_with_hemisphere']}")
    print(f"         Correlation: {conn['mean_correlation']:.3f}±{conn['std_correlation']:.3f}")
    print(f"         Consistency: {conn['frequency']}/{len(loaded_files)} files ({conn['frequency_percent']:.1f}%)")
    print()
    
    consistency_analysis.append({
        'network_pair': net_pair,
        'connection_type': conn_type,
        'frequency': conn['frequency'],
        'correlation': conn['mean_correlation'],
        'std_correlation': conn['std_correlation']
    })

print(f"Summary:")
print(f"  Within-network connections: {within_network}")
print(f"  Between-network connections: {between_network}")
print(f"  Unique network pairs: {len(network_connections)}")

# Hemisphere analysis
print(f"\n{'='*60}")
print("HEMISPHERE CONNECTIVITY ANALYSIS")
print(f"{'='*60}")

hemisphere_analysis = []
for conn in top_connections:
    hem_i = conn['hemisphere_i']
    hem_j = conn['hemisphere_j']
    
    if hem_i == hem_j:
        hemisphere_type = f"Within-{hem_i}"
    else:
        hemisphere_type = "Inter-hemispheric"
    
    hemisphere_analysis.append({
        'hemisphere_type': hemisphere_type,
        'connection': conn['connection_with_hemisphere'],
        'frequency': conn['frequency'],
        'correlation': conn['mean_correlation']
    })

# Count hemisphere connection types
from collections import Counter
hemisphere_counts = Counter([h['hemisphere_type'] for h in hemisphere_analysis])

print(f"Hemisphere connection breakdown:")
for hem_type, count in hemisphere_counts.items():
    print(f"  {hem_type}: {count} connections")
    
print(f"\nDetailed hemisphere connections:")
for h_analysis in hemisphere_analysis:
    print(f"  {h_analysis['hemisphere_type']}: {h_analysis['connection']}")
    print(f"    Freq: {h_analysis['frequency']}/{len(loaded_files)}, Corr: {h_analysis['correlation']:.3f}")

# Consistency by network analysis
consistency_df = pd.DataFrame(consistency_analysis)
network_summary = consistency_df.groupby(['network_pair', 'connection_type']).agg({
    'correlation': ['count', 'mean'],
    'frequency': 'mean',
    'std_correlation': 'mean'
}).round(3)

print(f"\nNetwork consistency analysis:")
print(network_summary)

# Reliability metrics
reliable_pairs = pair_stats_df[
    (pair_stats_df['frequency'] >= 0.5 * len(loaded_files)) & 
    (pair_stats_df['mean_correlation'] >= pair_stats_df['mean_correlation'].quantile(0.75))
]

print(f"\n🎯 RELIABILITY SUMMARY:")
print(f"📊 Total connections analyzed: {len(pair_stats_df)}")
print(f"📊 Connections in ≥50% files: {len(pair_stats_df[pair_stats_df['frequency'] >= 0.5 * len(loaded_files)])}")
print(f"📊 High-reliability connections (≥50% freq + top 25% corr): {len(reliable_pairs)}")
print(f"📊 Connections used in visualization: 10 most consistent")

In [ ]:
# Load the atlas file
import templateflow.api as tf
import nibabel as nib
import numpy as np
from nilearn.plotting import view_surf, plot_surf_roi
atlas_paths = tf.get(
    template='MNI152NLin2009cAsym', atlas='Schaefer2018',
    desc='100Parcels7Networks', extension='.nii.gz'
)

atlas_path = atlas_paths[0] if isinstance(atlas_paths, list) else atlas_paths
atlas_img = nib.load(atlas_path)
atlas_data = atlas_img.get_fdata()

# change values of this atlas from 0 to 100 corresponding to the 100 ROIs
atlas_data = np.where(atlas_data > 0, atlas_data, 100)
atlas_img = nib.Nifti1Image(atlas_data, atlas_img.affine)
# Load fsaverage5 surface
fsaverage = tf.get('fsaverage5', suffix='lh.surf.gii')
fsaverage_rh = tf.get('fsaverage5', suffix='rh.surf.gii')
fsaverage_lh = tf.get('fsaverage5', suffix='lh.surf.gii')
fsaverage_infl = tf.get('fsaverage5', suffix='lh.inflated.surf.gii')
fsaverage_infl_rh = tf.get('fsaverage5', suffix='rh.inflated.surf.gii')



view_surf(fsaverage_infl, atlas_img, hemi='left', title='Schaefer 100 Parcels Atlas - Left Hemisphere', cmap='tab20', colorbar=True)


In [ ]:
import numpy as np
data = np.load("fMRI_windows_analysis/fMRI_K10_seed500_most_significant.npz", allow_pickle=True)
data.files

In [ ]:
data["all_cluster_labels"]